# Bias

Visualizes directional opinion biases introduced by each LLM (Section 2): Bayesian intercepts by topic, original vs. transformed opinion scatter plot, and directly expressed LLM opinion vs. Bayesian intercept. Reads from `outputs/bayesian/` and `outputs/predict_opinion/`.

In [1]:
import os
os.chdir("../")

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import json
from src import utils
from IPython.display import clear_output
import numpy as np
from scipy import stats
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.legend_handler import HandlerBase
from PIL import Image

os.environ['PATH'] = f"{os.path.expanduser('~/.TinyTeX/bin/x86_64-linux')}:{os.environ['PATH']}"

sns.set_theme(context='paper', style='ticks', font_scale=1)

In [3]:
name="bias"
width_pt = 469
palette = sns.color_palette('husl', 5)

## Configuration

In [4]:
# model_name = "mistralai/Ministral-3-8B-Instruct-2512"
# model_name = "meta-llama/Llama-3.1-8B-Instruct"
model_name = "google/gemma-3-12b-it"
# model_name = "Qwen/Qwen3-8B"

dataset = "ukp"
# dataset = "semeval"

task = "writing"
# task = "improvement"

quantification_method = "centroid"

keep_flips = False
bayesian_suffix = "__keep_flips" if keep_flips else ""

# UKP topics
# topic = "abortion"
# topic = "cloning"
# topic = "death_penalty"
# topic = "gun_control"
# topic = "marijuana_legalization"
# topic = "minimum_wage"
# topic = "nuclear_energy"
# topic = "school_uniforms"

# SemEval topics
# topic = "atheism"
# topic = "acknowledging_climate_change"
topic = "feminism"
# topic = "hillary_clinton"
# topic = "abortion"
# topic = "donald_trump"

## Intercept by topic

In [5]:
if dataset == "semeval":
    topics = ["atheism", "acknowledging_climate_change", "feminism", "hillary_clinton", "abortion", "donald_trump"]
else:
    topics = ["abortion", "cloning", "death_penalty", "gun_control",
              "marijuana_legalization", "minimum_wage", "nuclear_energy", "school_uniforms"]

intercept_data = []
for t in topics:
    bayesian_file = f"outputs/bayesian/{dataset}__{task}__{model_name.replace('/', '_')}__{t}__{quantification_method}{bayesian_suffix}.json"
    with open(bayesian_file, 'r') as f:
        data = json.load(f)
    intercept = data['model_direction']['intercept']
    topic_text = t.replace('_', ' ')
    topic_text = topic_text.title() if t in ('donald_trump', 'hillary_clinton') else topic_text[0].upper() + topic_text[1:]
    intercept_data.append({
        'topic': topic_text,
        'mean': intercept['mean'],
        'lower_95': intercept['lower_95'],
        'upper_95': intercept['upper_95']
    })

intercept_df = pd.DataFrame(intercept_data)
intercept_df = intercept_df.sort_values('mean').reset_index(drop=True)


def wrap_label(label):
    if label == "Acknowledging climate change":
        return "Acknowledging\nclimate change"
    return label.replace(' ', '\n')


intercept_df['topic'] = intercept_df['topic'].map(wrap_label)

utils.latexify()
fig_width, fig_height = utils.get_fig_dim(width_pt, fraction=0.6)
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

bias_cmap = sns.color_palette("flare", as_cmap=True)
max_abs_mean = intercept_df['mean'].abs().max()
norm = plt.Normalize(vmin=0, vmax=max_abs_mean if max_abs_mean > 0 else 1)

ci_includes_zero = (intercept_df['lower_95'] <= 0) & (intercept_df['upper_95'] >= 0)

for i, row in intercept_df.iterrows():
    color = 'silver' if ci_includes_zero.iloc[i] else bias_cmap(norm(abs(row['mean'])))
    ax.errorbar(
        y=i, x=row['mean'],
        xerr=[[row['mean'] - row['lower_95']], [row['upper_95'] - row['mean']]],
        fmt='o', capsize=4, color=color,
        elinewidth=2, markersize=6, capthick=2,
    )

sns.despine(ax=ax)

ax.axvline(x=0, color='gray', linestyle='--', linewidth=1)
ax.set_yticks(range(len(intercept_df)))
ax.set_yticklabels(intercept_df['topic'], fontsize=7)
for label in ax.get_yticklabels():
    label.set_multialignment('center')
ax.set_ylabel("Topic")
ax.set_xlabel(r"Bias towards ``in favor''")

fig.tight_layout()
fig.savefig(f'figures/{name}__intercept_by_topic__{dataset}__{model_name.replace("/", "_")}{bayesian_suffix}.pdf', dpi=300)
plt.close()

## Original vs transformed scatter plot

In [6]:
predict_opinion_file = f"outputs/predict_opinion/predict_opinion__dataset={dataset}__task={task}__model={model_name.replace('/', '_')}__topic={topic}__quantification_method={quantification_method}.tsv"
predict_df = pd.read_csv(predict_opinion_file, sep="\t", dtype=str, quoting=3, on_bad_lines='warn')
predict_df['confidence_original'] = predict_df['confidence_original'].astype(float)
predict_df['confidence_transformed'] = predict_df['confidence_transformed'].astype(float)

average_by_original = True

filt_df = predict_df[(predict_df["confidence_original"] - 0.5) * (predict_df["confidence_transformed"] - 0.5) > 0]

if average_by_original:
    plot_df = filt_df.groupby('sentence_id').agg(
        confidence_original=('confidence_original', 'first'),
        confidence_transformed=('confidence_transformed', 'mean')
    ).reset_index()
else:
    plot_df = filt_df[['sentence_id', 'confidence_original', 'confidence_transformed']].copy()

utils.latexify()

fig_width, fig_height = utils.get_fig_dim(width_pt, fraction=0.6)
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

ax.scatter(plot_df['confidence_original'], plot_df['confidence_transformed'],
           s=20, alpha=0.3, color='silver', edgecolors='none')

ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1, zorder=0)

mask_below = plot_df['confidence_original'] < 0.5
mask_above = plot_df['confidence_original'] >= 0.5

fit_results = {}
for mask, label in [(mask_above, 'In favor'), (mask_below, 'Against')]:
    subset = plot_df[mask]
    if len(subset) < 2:
        continue
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        subset['confidence_original'], subset['confidence_transformed']
    )
    x_fit = np.linspace(subset['confidence_original'].min(), subset['confidence_original'].max(), 100)
    y_fit = slope * x_fit + intercept
    color = palette[0] if label == 'Against' else palette[2]
    ax.plot(subset['confidence_original'].mean(), subset['confidence_transformed'].mean(),
            marker='x', color=color, markersize=10, markeredgewidth=2.5, zorder=4)
    fit_results[label] = {'slope': slope, 'intercept': intercept, 'color': color}

ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.set_xlabel(r"Original opinion (human)")
ax.set_ylabel(r"Transformed opinion (LLM)")

sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(f'figures/{name}__scatter_original_vs_transformed__{dataset}__{model_name.replace("/", "_")}__{topic}.pdf', dpi=300)
plt.close()

## Directly expressed opinion vs bias intercept

In [5]:
utils.latexify()

fig_width, fig_height = utils.get_fig_dim(width_pt, fraction=0.6)
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

model_to_icon = {
    'mistralai_Ministral-3-8B-Instruct-2512': 'assets/icons/mistral-color.png',
    'meta-llama_Llama-3.1-8B-Instruct': 'assets/icons/meta-color.png',
    'google_gemma-3-12b-it': 'assets/icons/gemma-color.png',
    'Qwen_Qwen3-8B': 'assets/icons/qwen-color.png',
}

model_to_label = {
    'mistralai_Ministral-3-8B-Instruct-2512': 'Ministral',
    'meta-llama_Llama-3.1-8B-Instruct': 'Llama',
    'google_gemma-3-12b-it': 'Gemma',
    'Qwen_Qwen3-8B': 'Qwen',
}

def load_icon(path, target_size=80):
    img = Image.open(path).convert('RGBA')
    bbox = img.getbbox()
    if bbox:
        img = img.crop(bbox)
    w, h = img.size
    scale = target_size / max(w, h)
    new_w, new_h = round(w * scale), round(h * scale)
    img = img.resize((new_w, new_h), Image.LANCZOS)
    canvas = Image.new('RGBA', (target_size, target_size), (0, 0, 0, 0))
    canvas.paste(img, ((target_size - new_w) // 2, (target_size - new_h) // 2))
    img = canvas
    return np.array(img) / 255.0

scatter_data = []
bayesian_files = glob.glob(f'outputs/bayesian/{dataset}__{task}__*__{quantification_method}{bayesian_suffix}.json')

for bf in bayesian_files:
    basename = os.path.basename(bf).replace('.json', '')
    parts = basename.split('__')
    bf_model = parts[2]
    bf_topic = parts[3]

    opinion_file = f'outputs/measure_llm_opinion/measure_llm_opinion__dataset={dataset}__model={bf_model}__topic={bf_topic}__quantification_method={quantification_method}.tsv'
    if not os.path.exists(opinion_file):
        continue

    with open(bf, 'r') as f:
        bayesian_data = json.load(f)
    intercept_mean = bayesian_data['model_direction']['intercept']['mean']

    opinion_df = pd.read_csv(opinion_file, sep='\t', dtype=str, quoting=3, on_bad_lines='warn')
    opinion_df['confidence'] = opinion_df['confidence'].astype(float)
    avg_confidence = opinion_df['confidence'].mean()

    scatter_data.append({
        'model': bf_model,
        'topic': bf_topic,
        'opinion': avg_confidence,
        'intercept_mean': intercept_mean
    })

scatter_df = pd.DataFrame(scatter_data)

r, p = stats.pearsonr(scatter_df['opinion'], scatter_df['intercept_mean'])

x_min, x_max = scatter_df['opinion'].min(), scatter_df['opinion'].max()
y_min, y_max = scatter_df['intercept_mean'].min(), scatter_df['intercept_mean'].max()
x_padding = (x_max - x_min) * 0.2
y_padding = (y_max - y_min) * 0.3
ax.set_xlim([x_min - x_padding, x_max + x_padding])
ax.set_ylim([y_min - y_padding, y_max + y_padding])

icon_zoom = 0.13
icon_cache = {}
for _, row in scatter_df.iterrows():
    icon_path = model_to_icon.get(row['model'])
    if icon_path is None:
        continue
    if icon_path not in icon_cache:
        icon_cache[icon_path] = load_icon(icon_path)
    img = OffsetImage(icon_cache[icon_path], zoom=icon_zoom)
    ab = AnnotationBbox(img, (row['opinion'], row['intercept_mean']),
                        frameon=False)
    ax.add_artist(ab)

from matplotlib.patches import FancyBboxPatch

if dataset == 'semeval':
    highlight_topics = ['donald_trump', 'atheism', 'feminism']
    label_above_topics = {'donald_trump', 'feminism'}
    padding_x = 0.035
    padding_y = 0.01
    label_gap = 0.012
else:
    highlight_topics = ['gun_control', 'death_penalty', 'cloning']
    label_above_topics = {'gun_control', 'cloning'}
    padding_x = 0.035
    padding_y = 0.01
    label_gap = 0.008

box_colors = [palette[1], palette[3], palette[4]]

for ht, color in zip(highlight_topics, box_colors):
    topic_df = scatter_df[scatter_df['topic'] == ht]
    if len(topic_df) == 0:
        continue

    x_min_box, x_max_box = topic_df['opinion'].min(), topic_df['opinion'].max()
    y_min_box, y_max_box = topic_df['intercept_mean'].min(), topic_df['intercept_mean'].max()

    rect = FancyBboxPatch(
        (x_min_box - padding_x, y_min_box - padding_y),
        (x_max_box - x_min_box) + 2 * padding_x,
        (y_max_box - y_min_box) + 2 * padding_y,
        boxstyle="round,pad=0,rounding_size=0.01",
        linewidth=1,
        edgecolor=color,
        facecolor='none',
        linestyle='-',
        zorder=0
    )
    ax.add_patch(rect)

    topic_text = ht.replace('_', ' ')
    topic_text = topic_text.title() if ht in ('donald_trump', 'hillary_clinton') else topic_text[0].upper() + topic_text[1:]

    if ht in label_above_topics:
        ax.text((x_max_box + x_min_box)/2, y_max_box + padding_y + label_gap,
                topic_text,
                fontsize=10, va='center', ha='center', color=color)
    else:
        ax.text((x_max_box + x_min_box)/2, y_min_box - padding_y - label_gap,
                topic_text,
                fontsize=10, va='center', ha='center', color=color)

class IconHandler(HandlerBase):
    def __init__(self, icon_arr):
        self.icon_arr = icon_arr
        super().__init__()

    def create_artists(self, legend, orig_handle, xdescent, ydescent,
                       width, height, fontsize, trans):
        img = OffsetImage(self.icon_arr, zoom=0.15)
        ab = AnnotationBbox(img, (width / 2., height / 2.),
                            xycoords=trans, frameon=False)
        return [ab]

if p < 0.05:
    p_format = "< 0.05"
else:
    p_round = float(np.round(p, 3))
    p_format = f"= {p_round:.2f}"

ax.text(0.05, 0.95, f'$r = {r:.2f}$\n$p {p_format}$',
        transform=ax.transAxes, va='top', ha='left')

ax.axvline(x=0.5, color='gray', linestyle='--', linewidth=1)
ax.axhline(y=0.0, color='gray', linestyle='--', linewidth=1)

sns.despine(ax=ax)
ax.set_xlabel("Average directly expressed opinion")
ax.set_ylabel(r"Average bias towards ``in favor''")

fig.tight_layout()
fig.savefig(f'figures/{name}__opinion_vs_intercept__{dataset}{bayesian_suffix}.pdf', dpi=300)
plt.close()